# SNI-21 R0 Repaste Control

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ediprin/coffee-bean-detection/blob/agent/add-vadcp-pipeline/notebooks/SNI21_R0_Repaste_Control_Colab.ipynb)

Kontrol ini memakai checkpoint A0 yang sama dan **tidak melakukan training**. Cutout validation ditempel kembali pada gambar R0 validation di posisi dan ukuran bbox asal. Test tetap terkunci.

In [ ]:
from google.colab import drive
from pathlib import Path

if Path('/content/drive/MyDrive').is_dir():
    drive.flush_and_unmount()
drive.mount('/content/drive', force_remount=True)

import json, os, subprocess, sys
REPO = Path('/content/coffee-bean-detection')
BRANCH = 'agent/add-vadcp-pipeline'
URL = 'https://github.com/ediprin/coffee-bean-detection.git'
if not (REPO / '.git').is_dir():
    subprocess.run(['git', 'clone', '--branch', BRANCH, '--single-branch', URL, str(REPO)], check=True)
else:
    subprocess.run(['git', 'fetch', 'origin', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'checkout', BRANCH], cwd=REPO, check=True)
    subprocess.run(['git', 'pull', '--ff-only', 'origin', BRANCH], cwd=REPO, check=True)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-e', str(REPO)], check=True)
sys.path.insert(0, str(REPO / 'src'))
os.chdir(REPO)
import coffee_detector, torch
assert torch.cuda.is_available(), 'Aktifkan runtime T4 GPU.'
print('GPU:', torch.cuda.get_device_name(0))
print('IMPORT:', coffee_detector.__file__)
subprocess.run(['git', 'log', '-1', '--oneline'], cwd=REPO, check=True)

In [ ]:
DRIVE = Path('/content/drive/MyDrive')
SHORTCUTS = Path('/content/drive/.shortcut-targets-by-id')
SEARCH_ROOTS = [DRIVE, SHORTCUTS, Path('/content/drive/Shareddrives')]

def find_by_suffix(relative, label):
    suffix = Path(relative).parts
    direct = DRIVE / relative
    if direct.is_file():
        return direct
    matches = []
    for root in SEARCH_ROOTS:
        if not root.is_dir():
            continue
        for path in root.rglob(suffix[-1]):
            if path.is_file() and tuple(path.parts[-len(suffix):]) == suffix:
                matches.append(path)
    matches = sorted(set(matches))
    if not matches:
        raise FileNotFoundError(f'{label} tidak ditemukan: suffix={relative}')
    if len(matches) > 1:
        print(f'{label}: {len(matches)} kandidat; memakai {matches[0]}')
    return matches[0]

preferred_index = DRIVE / '02_RISET_DAN_PROYEK/Coffee_Bean_Detection/artifact_index.json'
index_matches = [preferred_index] if preferred_index.is_file() else []
if not index_matches:
    for root in SEARCH_ROOTS:
        if root.is_dir():
            index_matches.extend(path for path in root.rglob('artifact_index.json') if path.is_file())
if index_matches:
    PROJECT_INDEX = sorted(set(index_matches))[0]
    index = json.loads(PROJECT_INDEX.read_text(encoding='utf-8'))
    assert index['project'] == 'coffee-bean-detection'
    PROJECT_ROOT = PROJECT_INDEX.parent
    artifacts = index['artifacts']
    CHECKPOINT = PROJECT_ROOT / artifacts['a0_checkpoint']
    A0_ARCHIVE = PROJECT_ROOT / artifacts['a0_real_archive_optional']
    BENCHMARK_ROOT = PROJECT_ROOT / artifacts['density_benchmark_root']
    DENSITY_EVALUATION_ROOT = PROJECT_ROOT / artifacts['density_evaluation_output']
else:
    print('artifact_index.json belum tersinkron; memakai pencarian artefak langsung.')
    benchmark_summary = find_by_suffix('sni21-density-benchmark-v1/setup_core_summary.json', 'Summary benchmark')
    density_summary = find_by_suffix('sni21-density-evaluation-v1/density_evaluation_summary.json', 'Summary evaluasi density')
    CHECKPOINT = find_by_suffix('A0_seed42/weights/best.pt', 'Checkpoint A0')
    A0_ARCHIVE = find_by_suffix('sni21-vadcp-pilot-bundle/A0_real.tar', 'Archive A0')
    BENCHMARK_ROOT = benchmark_summary.parent
    DENSITY_EVALUATION_ROOT = density_summary.parent
    PROJECT_ROOT = BENCHMARK_ROOT.parent.parent
assert CHECKPOINT.is_file(), CHECKPOINT
assert A0_ARCHIVE.is_file(), A0_ARCHIVE
assert (BENCHMARK_ROOT / 'val_object_library/object_library.json').is_file()
assert (DENSITY_EVALUATION_ROOT / 'density_evaluation_summary.json').is_file()
R0_ROOT = Path('/content/sni21-fullscene-v1')
BENCHMARK_OUTPUT = Path('/content/sni21-r0-repaste-v1')
EVALUATION_OUTPUT = PROJECT_ROOT / 'experiments/sni21-r0-repaste-v1'
print('PROJECT:', PROJECT_ROOT)
print('OUTPUT :', EVALUATION_OUTPUT)

In [ ]:
from coffee_detector.archive_sni21_pilot import restore_real_a0_validation
restore_real_a0_validation(A0_ARCHIVE, R0_ROOT)
restore = json.loads((R0_ROOT / 'validation_restore.json').read_text(encoding='utf-8'))
assert restore['test_files_extracted'] == 0
assert restore['test_images_accessed'] is False
print(json.dumps(restore, indent=2))

In [ ]:
command = [
    sys.executable, '-u', '-m', 'coffee_detector.run_sni21_r0_repaste_control',
    '--checkpoint', str(CHECKPOINT),
    '--real-root', str(R0_ROOT),
    '--source-benchmark-root', str(BENCHMARK_ROOT),
    '--density-evaluation-root', str(DENSITY_EVALUATION_ROOT),
    '--benchmark-output-root', str(BENCHMARK_OUTPUT),
    '--evaluation-output-root', str(EVALUATION_OUTPUT),
    '--minimum-coverage', '0.85', '--device', '0', '--imgsz', '640',
    '--batch-size', '8', '--confidence', '0.001', '--nms-iou', '0.7',
    '--diagnostic-iou', '0.5', '--max-det', '300',
]
print('MENJALANKAN:', ' '.join(command), flush=True)
process = subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1)
for line in process.stdout:
    print(line, end='', flush=True)
assert process.wait() == 0, 'R0-repaste gagal; baca traceback di atas.'

In [ ]:
import pandas as pd
from IPython.display import display
path = EVALUATION_OUTPUT / 'r0_repaste_summary.json'
assert path.is_file(), path
summary = json.loads(path.read_text(encoding='utf-8'))
assert summary['training_executed'] is False
assert summary['test_images_accessed'] is False
table = pd.DataFrame(summary['rows'])
columns = ['map50_95', 'map50', 'precision', 'recall', 'macro_map50_95', 'bottom3_map50_95', 'worst_map50_95', 'proposal_recall_at_50', 'conditional_class_accuracy']
display(table.style.format({column: '{:.2%}' for column in columns}))
print(json.dumps(summary['attribution'], indent=2))
print('SUMMARY:', path)
print('Kirim tabel dan attribution ini. Jangan training model baru.')